In [0]:
%pip install pytest

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
test_code = """
import pytest
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ============================================================
# SPARK
# ============================================================

spark = SparkSession.builder.getOrCreate()

CATALOG = "`real-time-sentiment-catlog`"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.dbt_ychanna_silver"
GOLD = f"{CATALOG}.dbt_ychanna_gold"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_table(schema, table_name):
    return spark.table(f"{schema}.{table_name}")


def table_exists(schema, table_name):
    try:
        spark.sql(f"DESCRIBE TABLE {schema}.{table_name}")
        return True
    except Exception:
        return False


def assert_columns_present(df, columns, table_name):
    missing_columns = [column for column in columns if column not in df.columns]

    assert not missing_columns, (
        f"{table_name}: Missing expected columns: {missing_columns}"
    )


def assert_no_nulls(df, columns, table_name):

    condition = None

    for column in columns:

        current_condition = F.col(f"`{column}`").isNull()

        if condition is None:
            condition = current_condition
        else:
            condition = condition | current_condition

    null_count = df.filter(condition).count()

    assert null_count == 0, (
        f"{table_name}: Found {null_count} records "
        f"with NULL required fields: {columns}"
    )


def assert_no_negative_values(df, columns, table_name):

    condition = None

    for column in columns:

        current_condition = F.col(f"`{column}`") < 0

        if condition is None:
            condition = current_condition
        else:
            condition = condition | current_condition

    negative_count = df.filter(condition).count()

    assert negative_count == 0, (
        f"{table_name}: Found {negative_count} records "
        f"with negative values in {columns}"
    )


def assert_percentage_valid(df, column, table_name):

    invalid_count = df.filter(
        F.col(f"`{column}`").isNotNull() &
        (
            (F.col(f"`{column}`") < 0) |
            (F.col(f"`{column}`") > 100)
        )
    ).count()

    assert invalid_count == 0, (
        f"{table_name}: Found {invalid_count} records "
        f"with invalid {column} values outside 0-100"
    )


# ============================================================
# BASIC PYTEST CHECK
# ============================================================

def test_pytest_is_running():

    assert True


# ============================================================
# BRONZE TABLES
# ============================================================

BRONZE_TABLES = [
    "sentiments",
    "trends",
    "tweets",
    "user_metadata",
    "valid_tweets"
]


@pytest.mark.parametrize("table_name", BRONZE_TABLES)
def test_bronze_tables_exist(table_name):

    assert table_exists(
        BRONZE,
        table_name
    ), f"Bronze table does not exist: {table_name}"


@pytest.mark.parametrize("table_name", BRONZE_TABLES)
def test_bronze_tables_not_empty(table_name):

    df = get_table(BRONZE, table_name)

    assert df.count() > 0, (
        f"Bronze table is empty: {table_name}"
    )


# ============================================================
# BRONZE - SENTIMENTS
# ============================================================

def test_bronze_sentiments_required_fields():

    df = get_table(BRONZE, "sentiments")

    assert_columns_present(
        df,
        [
            "tweet_id",
            "topic_category",
            "tweet_timestamp"
        ],
        "bronze.sentiments"
    )


def test_bronze_sentiments_metrics_non_negative():

    df = get_table(BRONZE, "sentiments")

    assert_no_negative_values(
        df.select(
            F.col("impressions").cast("double").alias("impressions"),
            F.col("likes").cast("double").alias("likes"),
            F.col("engagement_count").cast("double").alias("engagement_count")
        ),
        [
            "impressions",
            "likes",
            "engagement_count"
        ],
        "bronze.sentiments"
    )


# ============================================================
# BRONZE - TRENDS
# ============================================================

def test_bronze_trends_required_fields():

    df = get_table(BRONZE, "trends")

    assert_columns_present(
        df,
        [
            "trend_timestamp",
            "topic_category",
            "country"
        ],
        "bronze.trends"
    )


def test_bronze_trends_metrics_non_negative():

    df = get_table(BRONZE, "trends")

    assert_no_negative_values(
        df.select(
            F.col("tweet_volume").cast("double").alias("tweet_volume"),
            F.col("mention_count").cast("double").alias("mention_count"),
            F.col("retweet_count").cast("double").alias("retweet_count"),
            F.col("impressions").cast("double").alias("impressions"),
            F.col("engagement_count").cast("double").alias("engagement_count")
        ),
        [
            "tweet_volume",
            "mention_count",
            "retweet_count",
            "impressions",
            "engagement_count"
        ],
        "bronze.trends"
    )


# ============================================================
# BRONZE - TWEETS
# ============================================================

def test_bronze_tweets_required_fields():

    df = get_table(BRONZE, "tweets")

    assert_columns_present(
        df,
        [
            "tweet_id",
            "user_id",
            "tweet_text",
            "timestamp"
        ],
        "bronze.tweets"
    )


def test_bronze_tweets_metrics_non_negative():

    df = get_table(BRONZE, "tweets")

    assert_no_negative_values(
        df.select(
            F.col("likes").cast("double").alias("likes"),
            F.col("retweets").cast("double").alias("retweets"),
            F.col("replies").cast("double").alias("replies"),
            F.col("impressions").cast("double").alias("impressions"),
            F.col("engagement").cast("double").alias("engagement")
        ),
        [
            "likes",
            "retweets",
            "replies",
            "impressions",
            "engagement"
        ],
        "bronze.tweets"
    )


# ============================================================
# BRONZE - USER METADATA
# ============================================================

def test_bronze_user_metadata_required_fields():

    df = get_table(BRONZE, "user_metadata")

    assert_columns_present(
        df,
        [
            "user_id",
            "country",
            "topic_category"
        ],
        "bronze.user_metadata"
    )


def test_bronze_user_metadata_metrics_non_negative():

    df = get_table(BRONZE, "user_metadata")

    assert_no_negative_values(
        df.select(
            F.col("followers_count").cast("double").alias("followers_count"),
            F.col("following_count").cast("double").alias("following_count"),
            F.col("likes_count").cast("double").alias("likes_count"),
            F.col("shares_count").cast("double").alias("shares_count"),
            F.col("posts_count").cast("double").alias("posts_count")
        ),
        [
            "followers_count",
            "following_count",
            "likes_count",
            "shares_count",
            "posts_count"
        ],
        "bronze.user_metadata"
    )


# ============================================================
# BRONZE - VALID TWEETS
# ============================================================

def test_bronze_valid_tweets_required_fields():

    df = get_table(BRONZE, "valid_tweets")

    assert_columns_present(
        df,
        [
            "tweet_id",
            "topic_category",
            "tweet_text",
            "tweet_timestamp"
        ],
        "bronze.valid_tweets"
    )


def test_bronze_valid_tweets_metrics_non_negative():

    df = get_table(BRONZE, "valid_tweets")

    assert_no_negative_values(
        df.select(
            F.col("impressions").cast("double").alias("impressions"),
            F.col("likes").cast("double").alias("likes"),
            F.col("retweets").cast("double").alias("retweets"),
            F.col("replies").cast("double").alias("replies"),
            F.col("engagement_count").cast("double").alias("engagement_count")
        ),
        [
            "impressions",
            "likes",
            "retweets",
            "replies",
            "engagement_count"
        ],
        "bronze.valid_tweets"
    )


# ============================================================
# SILVER TABLES
# ============================================================

SILVER_TABLES = [
    "silver_sentiments",
    "silver_trends",
    "silver_tweets",
    "silver_user_metadata",
    "silver_valid_tweets"
]


@pytest.mark.parametrize("table_name", SILVER_TABLES)
def test_silver_tables_exist(table_name):

    assert table_exists(
        SILVER,
        table_name
    ), f"Silver table does not exist: {table_name}"


@pytest.mark.parametrize("table_name", SILVER_TABLES)
def test_silver_tables_not_empty(table_name):

    df = get_table(SILVER, table_name)

    assert df.count() > 0, (
        f"Silver table is empty: {table_name}"
    )


# ============================================================
# SILVER - SENTIMENTS
# ============================================================

def test_silver_sentiments_required_fields():

    df = get_table(SILVER, "silver_sentiments")

    assert_no_nulls(
        df,
        [
            "tweet_id",
            "topic_category",
            "tweet_timestamp"
        ],
        "silver.silver_sentiments"
    )


def test_silver_sentiment_scores_valid():

    df = get_table(SILVER, "silver_sentiments")

    invalid = df.filter(
        (F.col("positive_score") < 0) |
        (F.col("positive_score") > 1) |
        (F.col("negative_score") < 0) |
        (F.col("negative_score") > 1) |
        (F.col("neutral_score") < 0) |
        (F.col("neutral_score") > 1) |
        (F.col("sentiment_score") < -1) |
        (F.col("sentiment_score") > 1)
    ).count()

    assert invalid == 0, (
        f"Found {invalid} invalid sentiment score records"
    )


def test_silver_sentiments_metrics_non_negative():

    df = get_table(SILVER, "silver_sentiments")

    assert_no_negative_values(
        df,
        [
            "impressions",
            "likes",
            "engagement_count"
        ],
        "silver.silver_sentiments"
    )


def test_silver_sentiment_tweet_id_unique():

    df = get_table(SILVER, "silver_sentiments")

    duplicates = (
        df.groupBy("tweet_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0, (
        f"Found {duplicates} duplicate tweet_ids"
    )


# ============================================================
# SILVER - TRENDS
# ============================================================

def test_silver_trends_required_fields():

    df = get_table(SILVER, "silver_trends")

    assert_no_nulls(
        df,
        [
            "trend_timestamp",
            "topic_category",
            "country"
        ],
        "silver.silver_trends"
    )


def test_silver_trends_metrics_non_negative():

    df = get_table(SILVER, "silver_trends")

    assert_no_negative_values(
        df,
        [
            "tweet_volume",
            "mention_count",
            "retweet_count",
            "impressions",
            "engagement_count"
        ],
        "silver.silver_trends"
    )


def test_silver_trend_score_valid():

    df = get_table(SILVER, "silver_trends")

    invalid = df.filter(
        (F.col("trend_score") < 0) |
        (F.col("trend_score") > 1)
    ).count()

    assert invalid == 0


def test_silver_sentiment_index_valid():

    df = get_table(SILVER, "silver_trends")

    invalid = df.filter(
        (F.col("sentiment_index") < -1) |
        (F.col("sentiment_index") > 1)
    ).count()

    assert invalid == 0


# ============================================================
# SILVER - TWEETS
# ============================================================

def test_silver_tweets_required_fields():

    df = get_table(SILVER, "silver_tweets")

    assert_no_nulls(
        df,
        [
            "tweet_id",
            "user_id",
            "tweet_text",
            "timestamp",
            "timestamp.1"
        ],
        "silver.silver_tweets"
    )


def test_silver_tweets_metrics_non_negative():

    df = get_table(SILVER, "silver_tweets")

    assert_no_negative_values(
        df,
        [
            "likes",
            "retweets",
            "replies",
            "impressions",
            "engagement"
        ],
        "silver.silver_tweets"
    )


def test_silver_tweet_id_unique():

    df = get_table(SILVER, "silver_tweets")

    duplicates = (
        df.groupBy("tweet_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0


# ============================================================
# SILVER - USER METADATA
# ============================================================

def test_silver_user_metadata_required_fields():

    df = get_table(SILVER, "silver_user_metadata")

    assert_no_nulls(
        df,
        [
            "user_id",
            "country",
            "topic_category"
        ],
        "silver.silver_user_metadata"
    )


def test_silver_user_metadata_metrics_non_negative():

    df = get_table(SILVER, "silver_user_metadata")

    assert_no_negative_values(
        df,
        [
            "followers_count",
            "following_count",
            "likes_count",
            "shares_count",
            "posts_count"
        ],
        "silver.silver_user_metadata"
    )


def test_silver_user_id_unique():

    df = get_table(SILVER, "silver_user_metadata")

    duplicates = (
        df.groupBy("user_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0


# ============================================================
# SILVER - VALID TWEETS
# ============================================================

def test_silver_valid_tweets_required_fields():

    df = get_table(SILVER, "silver_valid_tweets")

    assert_no_nulls(
        df,
        [
            "tweet_id",
            "topic_category",
            "tweet_text",
            "tweet_timestamp"
        ],
        "silver.silver_valid_tweets"
    )


def test_silver_valid_tweets_metrics_non_negative():

    df = get_table(SILVER, "silver_valid_tweets")

    assert_no_negative_values(
        df,
        [
            "impressions",
            "likes",
            "retweets",
            "replies",
            "engagement_count"
        ],
        "silver.silver_valid_tweets"
    )


def test_silver_valid_tweet_sentiment_valid():

    df = get_table(SILVER, "silver_valid_tweets")

    invalid = df.filter(
        (F.col("sentiment_score") < -1) |
        (F.col("sentiment_score") > 1)
    ).count()

    assert invalid == 0


def test_silver_valid_tweet_id_unique():

    df = get_table(SILVER, "silver_valid_tweets")

    duplicates = (
        df.groupBy("tweet_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0


# ============================================================
# GOLD TABLES
# ============================================================

GOLD_TABLES = [
    "gold_sentiment_engagement",
    "gold_sentiment_summary",
    "gold_sentiment_topic",
    "gold_sentiment_trends",
    "gold_trend_country",
    "gold_trend_daily",
    "gold_trend_summary",
    "gold_tweet_summary",
    "gold_tweet_top",
    "gold_user_metadata_country",
    "gold_user_metadata_verified_users",
    "gold_valid_tweet_summary"
]


@pytest.mark.parametrize("table_name", GOLD_TABLES)
def test_gold_tables_exist(table_name):

    assert table_exists(
        GOLD,
        table_name
    ), f"Gold table does not exist: {table_name}"


@pytest.mark.parametrize("table_name", GOLD_TABLES)
def test_gold_tables_not_empty(table_name):

    df = get_table(GOLD, table_name)

    assert df.count() > 0, (
        f"Gold table is empty: {table_name}"
    )


# ============================================================
# GOLD - SENTIMENT ENGAGEMENT
# ============================================================

def test_gold_sentiment_engagement_quality():

    df = get_table(
        GOLD,
        "gold_sentiment_engagement"
    )

    assert_no_nulls(
        df,
        [
            "sentiment_category",
            "total_tweets"
        ],
        "gold.gold_sentiment_engagement"
    )

    assert_no_negative_values(
        df,
        [
            "total_tweets",
            "total_impressions",
            "total_likes",
            "total_engagement",
            "avg_impressions_per_tweet",
            "avg_likes_per_tweet",
            "avg_engagement_per_tweet"
        ],
        "gold.gold_sentiment_engagement"
    )


# ============================================================
# GOLD - SENTIMENT SUMMARY
# ============================================================

def test_gold_sentiment_summary_quality():

    df = get_table(
        GOLD,
        "gold_sentiment_summary"
    )

    assert_no_negative_values(
        df,
        [
            "total_tweets",
            "total_impressions",
            "total_likes",
            "total_engagement",
            "avg_engagement_per_tweet",
            "avg_impressions_per_tweet"
        ],
        "gold.gold_sentiment_summary"
    )

    invalid = df.filter(
        (F.col("avg_sentiment_score") < -1) |
        (F.col("avg_sentiment_score") > 1) |
        (F.col("avg_positive_score") < 0) |
        (F.col("avg_positive_score") > 1) |
        (F.col("avg_negative_score") < 0) |
        (F.col("avg_negative_score") > 1) |
        (F.col("avg_neutral_score") < 0) |
        (F.col("avg_neutral_score") > 1)
    ).count()

    assert invalid == 0


# ============================================================
# GOLD - SENTIMENT TOPIC
# ============================================================

def test_gold_sentiment_topic_quality():

    df = get_table(
        GOLD,
        "gold_sentiment_topic"
    )

    assert_no_nulls(
        df,
        ["topic_category"],
        "gold.gold_sentiment_topic"
    )

    assert_no_negative_values(
        df,
        [
            "total_tweets",
            "total_impressions",
            "total_likes",
            "total_engagement",
            "avg_engagement_per_tweet",
            "avg_impressions_per_tweet"
        ],
        "gold.gold_sentiment_topic"
    )

    invalid = df.filter(
        (F.col("avg_sentiment_score") < -1) |
        (F.col("avg_sentiment_score") > 1) |
        (F.col("avg_positive_score") < 0) |
        (F.col("avg_positive_score") > 1) |
        (F.col("avg_negative_score") < 0) |
        (F.col("avg_negative_score") > 1) |
        (F.col("avg_neutral_score") < 0) |
        (F.col("avg_neutral_score") > 1)
    ).count()

    assert invalid == 0


# ============================================================
# GOLD - SENTIMENT TRENDS
# ============================================================

def test_gold_sentiment_trends_quality():

    df = get_table(
        GOLD,
        "gold_sentiment_trends"
    )

    assert_no_nulls(
        df,
        ["sentiment_date"],
        "gold.gold_sentiment_trends"
    )

    assert_no_negative_values(
        df,
        [
            "total_tweets",
            "total_impressions",
            "total_likes",
            "total_engagement",
            "avg_engagement_per_tweet"
        ],
        "gold.gold_sentiment_trends"
    )

    invalid = df.filter(
        (F.col("avg_sentiment_score") < -1) |
        (F.col("avg_sentiment_score") > 1) |
        (F.col("avg_positive_score") < 0) |
        (F.col("avg_positive_score") > 1) |
        (F.col("avg_negative_score") < 0) |
        (F.col("avg_negative_score") > 1) |
        (F.col("avg_neutral_score") < 0) |
        (F.col("avg_neutral_score") > 1)
    ).count()

    assert invalid == 0


# ============================================================
# GOLD - TREND COUNTRY
# ============================================================

def test_gold_trend_country_quality():

    df = get_table(
        GOLD,
        "gold_trend_country"
    )

    assert_no_nulls(
        df,
        ["country"],
        "gold.gold_trend_country"
    )

    assert_no_negative_values(
        df,
        [
            "trend_record_count",
            "total_topics",
            "total_tweet_volume",
            "total_mentions",
            "total_retweets",
            "total_impressions",
            "total_engagement"
        ],
        "gold.gold_trend_country"
    )

    assert_percentage_valid(
        df,
        "engagement_rate_pct",
        "gold.gold_trend_country"
    )

    invalid = df.filter(
        (F.col("avg_trend_score") < 0) |
        (F.col("avg_trend_score") > 1) |
        (F.col("avg_sentiment_index") < -1) |
        (F.col("avg_sentiment_index") > 1)
    ).count()

    assert invalid == 0


# ============================================================
# GOLD - TREND DAILY
# ============================================================

def test_gold_trend_daily_quality():

    df = get_table(
        GOLD,
        "gold_trend_daily"
    )

    assert_no_nulls(
        df,
        ["trend_date"],
        "gold.gold_trend_daily"
    )

    assert_no_negative_values(
        df,
        [
            "trend_record_count",
            "total_topics",
            "total_countries",
            "total_tweet_volume",
            "total_mentions",
            "total_retweets",
            "total_impressions",
            "total_engagement",
            "avg_trend_score"
        ],
        "gold.gold_trend_daily"
    )

    invalid = df.filter(
        (F.col("avg_trend_score") < 0) |
        (F.col("avg_trend_score") > 1)
    ).count()

    assert invalid == 0


# ============================================================
# GOLD - TREND SUMMARY
# ============================================================

def test_gold_trend_summary_quality():

    df = get_table(
        GOLD,
        "gold_trend_summary"
    )

    assert_no_negative_values(
        df,
        [
            "total_trend_records",
            "total_topics",
            "total_countries",
            "total_tweet_volume",
            "total_mentions",
            "total_retweets",
            "total_impressions",
            "total_engagement"
        ],
        "gold.gold_trend_summary"
    )

    invalid = df.filter(
        (F.col("avg_trend_score") < 0) |
        (F.col("avg_trend_score") > 1) |
        (F.col("avg_sentiment_index") < -1) |
        (F.col("avg_sentiment_index") > 1)
    ).count()

    assert invalid == 0


# ============================================================
# GOLD - TWEET SUMMARY
# ============================================================

def test_gold_tweet_summary_quality():

    df = get_table(
        GOLD,
        "gold_tweet_summary"
    )

    assert_no_negative_values(
        df,
        [
            "total_tweets",
            "total_users",
            "total_likes",
            "total_retweets",
            "total_replies",
            "total_impressions",
            "total_engagement",
            "avg_likes_per_tweet",
            "avg_retweets_per_tweet",
            "avg_replies_per_tweet"
        ],
        "gold.gold_tweet_summary"
    )


# ============================================================
# GOLD - TOP TWEETS
# ============================================================

def test_gold_tweet_top_quality():

    df = get_table(
        GOLD,
        "gold_tweet_top"
    )

    assert_no_nulls(
        df,
        [
            "tweet_id",
            "user_id",
            "tweet_text",
            "tweet_timestamp"
        ],
        "gold.gold_tweet_top"
    )

    assert_no_negative_values(
        df,
        [
            "likes",
            "retweets",
            "replies",
            "impressions",
            "engagement"
        ],
        "gold.gold_tweet_top"
    )

    invalid = df.filter(
        F.col("engagement_rate_pct").isNull() |
        (F.col("engagement_rate_pct") < 0) |
        (F.col("impressions") <= 0) |
        (
            F.abs(
                F.col("engagement_rate_pct").cast("double") -
                F.round(
                    (F.col("engagement") * F.lit(100.0)) /
                    F.col("impressions"),
                    2
                )
            ) > 0.01
        )
    ).count()

    assert invalid == 0, (
        f"gold.gold_tweet_top: Found {invalid} records with invalid engagement_rate_pct values"
    )


# ============================================================
# GOLD - USER METADATA COUNTRY
# ============================================================

def test_gold_user_metadata_country_quality():

    df = get_table(
        GOLD,
        "gold_user_metadata_country"
    )

    assert_no_nulls(
        df,
        ["country"],
        "gold.gold_user_metadata_country"
    )

    assert_no_negative_values(
        df,
        [
            "total_users",
            "verified_users",
            "unverified_users",
            "total_followers",
            "total_following",
            "total_likes",
            "total_shares",
            "total_posts",
            "avg_followers"
        ],
        "gold.gold_user_metadata_country"
    )


# ============================================================
# GOLD - VERIFIED USERS
# ============================================================

def test_gold_verified_users_quality():

    df = get_table(
        GOLD,
        "gold_user_metadata_verified_users"
    )

    assert_no_nulls(
        df,
        ["verified"],
        "gold.gold_user_metadata_verified_users"
    )

    assert_no_negative_values(
        df,
        [
            "total_users",
            "total_followers",
            "total_following",
            "total_likes",
            "total_shares",
            "total_posts",
            "avg_followers",
            "avg_following",
            "avg_likes"
        ],
        "gold.gold_user_metadata_verified_users"
    )


# ============================================================
# GOLD - VALID TWEET SUMMARY
# ============================================================

def test_gold_valid_tweet_summary_quality():

    df = get_table(
        GOLD,
        "gold_valid_tweet_summary"
    )

    assert_no_negative_values(
        df,
        [
            "total_valid_tweets",
            "total_topics",
            "total_impressions",
            "total_likes",
            "total_retweets",
            "total_replies",
            "total_engagement",
            "avg_impressions_per_tweet",
            "avg_likes_per_tweet",
            "avg_retweets_per_tweet"
        ],
        "gold.gold_valid_tweet_summary"
    )
"""

with open("/tmp/test_sentiment_data_quality.py", "w") as f:
    f.write(test_code)

print("Corrected PyTest file created successfully")


Corrected PyTest file created successfully


In [0]:
import importlib
import pytest
import sys
from pathlib import Path

sys.dont_write_bytecode = True

test_file = Path("/tmp/test_sentiment_data_quality.py")
test_code = test_file.read_text()

if 'CATALOG = "`real-time-sentiment-catlog`"' not in test_code:
    test_code = test_code.replace(
        'CATALOG = "real-time-sentiment-catlog"',
        'CATALOG = "`real-time-sentiment-catlog`"'
    )
    test_file.write_text(test_code)

sys.modules.pop("test_sentiment_data_quality", None)
importlib.invalidate_caches()

result = pytest.main([
    "-v",
    "-p", "no:cacheprovider",
    "/tmp/test_sentiment_data_quality.py"
])

if result != 0:
    raise Exception(
        f"DATA QUALITY TESTS FAILED. Pytest exit code: {result}"
    )

print("========================================")
print("ALL DATA QUALITY TESTS PASSED")
print("========================================")


============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-8.3.5, pluggy-1.5.0 -- /local_disk0/.ephemeral_nfs/envs/pythonEnv-f422d480-1879-4530-a94f-e295eded8d0f/bin/python
rootdir: /tmp
plugins: langsmith-0.6.1, anyio-4.7.0
collecting ... collected 85 items

../../../../../tmp/test_sentiment_data_quality.py::test_pytest_is_running PASSED [  1%]
../../../../../tmp/test_sentiment_data_quality.py::test_bronze_tables_exist[sentiments] PASSED [  2%]
../../../../../tmp/test_sentiment_data_quality.py::test_bronze_tables_exist[trends] PASSED [  3%]
../../../../../tmp/test_sentiment_data_quality.py::test_bronze_tables_exist[tweets] PASSED [  4%]
../../../../../tmp/test_sentiment_data_quality.py::test_bronze_tables_exist[user_metadata] PASSED [  5%]
../../../../../tmp/test_sentiment_data_quality.py::test_bronze_tables_exist[valid_tweets] PASSED [  7%]
../../../../../tmp/test_sentiment_data_quality.py::test_bronze_tables_not_empty[sen